In [1]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [2]:
import json

In [3]:
from models import Ride, ride_deserializer

In [4]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='green-trips-console',
    value_deserializer=ride_deserializer
)

In [5]:
record = next(consumer)
print(record.value)

Ride(PULocationID=247, DOLocationID=69, passenger_count=1, trip_distance=0.7, tip_amount=1.7, total_amount=10.0, lpep_pickup_datetime=1759278107000, lpep_dropoff_datetime=1759278277000)


In [6]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [ ]:
CREATE TABLE green_trips (
    PULocationID INTEGER,
    DOLocationID INTEGER,
    passenger_count INTEGER,
    trip_distance DOUBLE PRECISION,
    tip_amount DOUBLE PRECISION,
    total_amount DOUBLE PRECISION,
    lpep_pickup_datetime TIMESTAMP,
    lpep_dropoff_datetime TIMESTAMP

);

In [9]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)
    cur.execute(
        """INSERT INTO green_trips
           (PULocationID, DOLocationID, passenger_count,
           trip_distance, tip_amount, total_amount, lpep_pickup_datetime, lpep_dropoff_datetime)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID, ride.passenger_count,
         ride.trip_distance, ride.tip_amount, ride.total_amount, pickup_dt, dropoff_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...


UndefinedColumn: column "lpep_pickup_datetime" of relation "green_trips" does not exist
LINE 3: ...         trip_distance, tip_amount, total_amount, lpep_picku...
                                                             ^
